# Расчетно-графическая работа
## Предметная область: «Туристическая фирма»

В работе реализована реляционная база данных туристической фирмы в PostgreSQL. Приложение создано в Jupyter Notebook с использованием SQLAlchemy ORM.

База данных хранит сведения о клиентах, странах, климате, отелях, путевках, фиксированных скидках и заказах. Стоимость продажи рассчитывается с учетом цены путевки и скидки.

## 1. Импорт библиотек

In [1]:
from datetime import date
from decimal import Decimal

import pandas as pd
from IPython.display import display
from sqlalchemy import (
    BigInteger, CheckConstraint, Column, Date, ForeignKey, Numeric,
    SmallInteger, String, Text, UniqueConstraint, create_engine, text
)
from sqlalchemy.engine import URL
from sqlalchemy.exc import SQLAlchemyError
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

print("Библиотеки успешно импортированы")

Библиотеки успешно импортированы


## 2. Подключение к PostgreSQL

Перед запуском создайте базу данных `travel_agency_db`. При необходимости измените пароль пользователя PostgreSQL в переменной `DB_PASSWORD`.

In [2]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "127.0.0.1"
DB_PORT = 5432
DB_NAME = "travel_agency_db"

DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

try:
    engine = create_engine(DATABASE_URL, echo=False)
    with engine.connect() as connection:
        database_name, user_name = connection.execute(
            text("SELECT current_database(), current_user;")
        ).one()
    print(f"Подключение к PostgreSQL выполнено успешно. База: {database_name}; пользователь: {user_name}")
except SQLAlchemyError as error:
    print("Ошибка подключения к PostgreSQL:")
    print(error)

Подключение к PostgreSQL выполнено успешно. База: travel_agency_db; пользователь: postgres


## 3. Описание ORM-моделей

Названия таблиц и полей соответствуют физической модели базы данных.

In [3]:
Base = declarative_base()


class Client(Base):
    __tablename__ = "client"

    client_id = Column(BigInteger, primary_key=True, autoincrement=True)
    last_name = Column(String(80), nullable=False)
    first_name = Column(String(80), nullable=False)
    middle_name = Column(String(80), nullable=True)
    address = Column(Text, nullable=True)
    phone = Column(String(30), nullable=True, unique=True)

    orders = relationship("Order", back_populates="client")

    def __repr__(self):
        return f"<Client(client_id={self.client_id}, full_name='{self.last_name} {self.first_name} {self.middle_name or ''}')>"


class Climate(Base):
    __tablename__ = "climate"

    climate_id = Column(BigInteger, primary_key=True, autoincrement=True)
    name = Column(String(100), nullable=False, unique=True)
    description = Column(Text, nullable=True)

    countries = relationship("Country", back_populates="climate")

    def __repr__(self):
        return f"<Climate(climate_id={self.climate_id}, name='{self.name}')>"


class Country(Base):
    __tablename__ = "country"

    country_id = Column(BigInteger, primary_key=True, autoincrement=True)
    name = Column(String(120), nullable=False, unique=True)
    climate_id = Column(BigInteger, ForeignKey("climate.climate_id"), nullable=False)

    climate = relationship("Climate", back_populates="countries")
    packages = relationship("TravelPackage", back_populates="country")

    def __repr__(self):
        return f"<Country(country_id={self.country_id}, name='{self.name}')>"


class Hotel(Base):
    __tablename__ = "hotel"

    hotel_id = Column(BigInteger, primary_key=True, autoincrement=True)
    name = Column(String(200), nullable=False)
    category = Column(SmallInteger, nullable=False)
    address = Column(Text, nullable=True)

    packages = relationship("TravelPackage", back_populates="hotel")

    __table_args__ = (
        CheckConstraint("category BETWEEN 1 AND 5", name="check_hotel_category"),
    )

    def __repr__(self):
        return f"<Hotel(hotel_id={self.hotel_id}, name='{self.name}', category={self.category})>"


class TravelPackage(Base):
    __tablename__ = "travel_package"

    package_id = Column(BigInteger, primary_key=True, autoincrement=True)
    country_id = Column(BigInteger, ForeignKey("country.country_id"), nullable=False)
    hotel_id = Column(BigInteger, ForeignKey("hotel.hotel_id"), nullable=False)
    price = Column(Numeric(12, 2), nullable=False)
    duration_weeks = Column(SmallInteger, nullable=False)

    country = relationship("Country", back_populates="packages")
    hotel = relationship("Hotel", back_populates="packages")
    orders = relationship("Order", back_populates="travel_package")

    __table_args__ = (
        CheckConstraint("price > 0", name="check_package_price_positive"),
        CheckConstraint("duration_weeks IN (1, 2, 4)", name="check_package_duration"),
        UniqueConstraint("country_id", "hotel_id", "duration_weeks", name="unique_package_variant"),
    )

    def __repr__(self):
        return f"<TravelPackage(package_id={self.package_id}, duration_weeks={self.duration_weeks}, price={self.price})>"


class Discount(Base):
    __tablename__ = "discount"

    discount_id = Column(BigInteger, primary_key=True, autoincrement=True)
    type = Column(String(80), nullable=False, unique=True)
    percent = Column(SmallInteger, nullable=False)

    orders = relationship("Order", back_populates="discount")

    __table_args__ = (
        CheckConstraint("percent BETWEEN 0 AND 100", name="check_discount_percent"),
    )

    def __repr__(self):
        return f"<Discount(discount_id={self.discount_id}, type='{self.type}', percent={self.percent})>"


class Order(Base):
    __tablename__ = "orders"

    order_id = Column(BigInteger, primary_key=True, autoincrement=True)
    client_id = Column(BigInteger, ForeignKey("client.client_id"), nullable=False)
    package_id = Column(BigInteger, ForeignKey("travel_package.package_id"), nullable=False)
    discount_id = Column(BigInteger, ForeignKey("discount.discount_id"), nullable=True)
    departure_date = Column(Date, nullable=False)

    client = relationship("Client", back_populates="orders")
    travel_package = relationship("TravelPackage", back_populates="orders")
    discount = relationship("Discount", back_populates="orders")

    def __repr__(self):
        return f"<Order(order_id={self.order_id}, client_id={self.client_id}, package_id={self.package_id})>"


print("ORM-модели успешно описаны")

ORM-модели успешно описаны


## 4. Создание таблиц

Перед повторным запуском удаляются созданные ранее представления, функции, процедуры и триггеры. После этого таблицы пересоздаются через SQLAlchemy ORM.

In [4]:
try:
    with engine.begin() as connection:
        connection.execute(text("DROP VIEW IF EXISTS order_payment_view CASCADE;"))
        connection.execute(text("DROP PROCEDURE IF EXISTS assign_discount_to_order(BIGINT, BIGINT);"))
        connection.execute(text("DROP FUNCTION IF EXISTS calculate_order_total(NUMERIC, INTEGER);"))
        connection.execute(text("""
            DO $$
            BEGIN
                IF to_regclass('public.travel_package') IS NOT NULL THEN
                    DROP TRIGGER IF EXISTS check_hotel_country_trigger ON travel_package;
                END IF;
            END $$;
        """))
        connection.execute(text("DROP FUNCTION IF EXISTS check_hotel_country_consistency() CASCADE;"))

    Base.metadata.drop_all(engine)
    Base.metadata.create_all(engine)
    print("Таблицы успешно созданы в базе данных")
except SQLAlchemyError as error:
    print("Ошибка при создании таблиц:")
    print(error)

Таблицы успешно созданы в базе данных


## 5. Создание сессии и функции вывода данных

In [5]:
SessionLocal = sessionmaker(bind=engine)


def show_query(sql):
    with engine.connect() as connection:
        return pd.read_sql(text(sql), connection)


print("Сессия и функция show_query успешно созданы")

Сессия и функция show_query успешно созданы


## 6. Заполнение базы данных тестовыми данными

В предоставленной физической модели заказ содержит одну ссылку на скидку. Чтобы сохранить эту структуру и учесть возможность суммирования фиксированных скидок, комбинированные скидки представлены отдельными записями справочника `discount`.

In [6]:
try:
    session = SessionLocal()

    climates = [
        Climate(name="Средиземноморский", description="Мягкая зима и теплое сухое лето"),
        Climate(name="Тропический", description="Высокая температура и влажность в течение года"),
        Climate(name="Умеренный северный", description="Прохладное лето и холодная снежная зима"),
    ]
    session.add_all(climates)
    session.commit()

    countries = [
        Country(name="Турция", climate_id=climates[0].climate_id),
        Country(name="Греция", climate_id=climates[0].climate_id),
        Country(name="Таиланд", climate_id=climates[1].climate_id),
        Country(name="Финляндия", climate_id=climates[2].climate_id),
    ]
    session.add_all(countries)
    session.commit()

    hotels = [
        Hotel(name="Sunrise Resort", category=4, address="Анталья, побережье Лара"),
        Hotel(name="Aegean Pearl", category=5, address="о. Родос, набережная Элли"),
        Hotel(name="Siam Garden", category=4, address="Пхукет, район Карон"),
        Hotel(name="Northern Lights Hotel", category=3, address="Рованиеми, центр города"),
    ]
    session.add_all(hotels)
    session.commit()

    packages = [
        TravelPackage(country_id=countries[0].country_id, hotel_id=hotels[0].hotel_id, price=Decimal("65000.00"), duration_weeks=1),
        TravelPackage(country_id=countries[0].country_id, hotel_id=hotels[0].hotel_id, price=Decimal("105000.00"), duration_weeks=2),
        TravelPackage(country_id=countries[1].country_id, hotel_id=hotels[1].hotel_id, price=Decimal("78000.00"), duration_weeks=1),
        TravelPackage(country_id=countries[1].country_id, hotel_id=hotels[1].hotel_id, price=Decimal("128000.00"), duration_weeks=2),
        TravelPackage(country_id=countries[2].country_id, hotel_id=hotels[2].hotel_id, price=Decimal("165000.00"), duration_weeks=2),
        TravelPackage(country_id=countries[2].country_id, hotel_id=hotels[2].hotel_id, price=Decimal("295000.00"), duration_weeks=4),
        TravelPackage(country_id=countries[3].country_id, hotel_id=hotels[3].hotel_id, price=Decimal("90000.00"), duration_weeks=1),
    ]
    session.add_all(packages)
    session.commit()

    discounts = [
        Discount(type="Постоянный клиент", percent=5),
        Discount(type="Раннее бронирование", percent=10),
        Discount(type="Постоянный клиент + раннее бронирование", percent=15),
        Discount(type="Семейная покупка", percent=7),
        Discount(type="Семейная покупка + раннее бронирование", percent=17),
    ]
    session.add_all(discounts)
    session.commit()

    clients = [
        Client(last_name="Иванова", first_name="Анна", middle_name="Сергеевна", address="г. Омск, ул. Ленина, д. 10", phone="+7-900-111-22-33"),
        Client(last_name="Петров", first_name="Максим", middle_name="Олегович", address="г. Омск, ул. Мира, д. 25", phone="+7-900-222-33-44"),
        Client(last_name="Соколова", first_name="Елена", middle_name="Андреевна", address="г. Омск, пр. Карла Маркса, д. 8", phone="+7-900-333-44-55"),
        Client(last_name="Кузнецов", first_name="Игорь", middle_name="Павлович", address="г. Омск, ул. Герцена, д. 17", phone="+7-900-444-55-66"),
    ]
    session.add_all(clients)
    session.commit()

    orders = [
        Order(client_id=clients[0].client_id, package_id=packages[0].package_id, discount_id=discounts[2].discount_id, departure_date=date(2026, 7, 10)),
        Order(client_id=clients[0].client_id, package_id=packages[0].package_id, discount_id=discounts[2].discount_id, departure_date=date(2026, 7, 10)),
        Order(client_id=clients[1].client_id, package_id=packages[3].package_id, discount_id=discounts[1].discount_id, departure_date=date(2026, 8, 5)),
        Order(client_id=clients[2].client_id, package_id=packages[4].package_id, discount_id=None, departure_date=date(2026, 9, 12)),
        Order(client_id=clients[3].client_id, package_id=packages[6].package_id, discount_id=discounts[0].discount_id, departure_date=date(2026, 12, 20)),
        Order(client_id=clients[2].client_id, package_id=packages[2].package_id, discount_id=discounts[3].discount_id, departure_date=date(2026, 6, 15)),
    ]
    session.add_all(orders)
    session.commit()

    print("Тестовые данные успешно добавлены")
except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка при добавлении тестовых данных:")
    print(error)
finally:
    session.close()

Тестовые данные успешно добавлены


## 7. Просмотр всех данных после добавления

In [7]:
tables = {
    "client": "client_id",
    "climate": "climate_id",
    "country": "country_id",
    "hotel": "hotel_id",
    "travel_package": "package_id",
    "discount": "discount_id",
    "orders": "order_id",
}

for table_name, order_column in tables.items():
    print(f"\nТаблица: {table_name}")
    display(show_query(f"SELECT * FROM {table_name} ORDER BY {order_column};"))


Таблица: client


,client_id,last_name,first_name,middle_name,address,phone
0,1,Иванова,Анна,Сергеевна,"г. Омск, ул. Ленина, д. 10",+7-900-111-22-33
1,2,Петров,Максим,Олегович,"г. Омск, ул. Мира, д. 25",+7-900-222-33-44
2,3,Соколова,Елена,Андреевна,"г. Омск, пр. Карла Маркса, д. 8",+7-900-333-44-55
3,4,Кузнецов,Игорь,Павлович,"г. Омск, ул. Герцена, д. 17",+7-900-444-55-66



Таблица: climate


,climate_id,name,description
0,1,Средиземноморский,Мягкая зима и теплое сухое лето
1,2,Тропический,Высокая температура и влажность в течение года
2,3,Умеренный северный,Прохладное лето и холодная снежная зима



Таблица: country


...

,package_id,country_id,hotel_id,price,duration_weeks
0,1,1,1,65000.0,1
1,2,1,1,105000.0,2
2,3,2,2,78000.0,1
3,4,2,2,128000.0,2
4,5,3,3,165000.0,2
5,6,3,3,295000.0,4
6,7,4,4,90000.0,1



Таблица: discount


,discount_id,type,percent
0,1,Постоянный клиент,5
1,2,Раннее бронирование,10
2,3,Постоянный клиент + раннее бронирование,15
3,4,Семейная покупка,7
4,5,Семейная покупка + раннее бронирование,17



Таблица: orders


,order_id,client_id,package_id,discount_id,departure_date
0,1,1,1,3.0,2026-07-10
1,2,1,1,3.0,2026-07-10
2,3,2,4,2.0,2026-08-05
3,4,3,5,NaN,2026-09-12
4,5,4,7,1.0,2026-12-20
5,6,3,3,4.0,2026-06-15


### Объединенный вывод заказов

In [8]:
show_query("""
SELECT
    o.order_id,
    c.last_name || ' ' || c.first_name || ' ' || COALESCE(c.middle_name, '') AS client_full_name,
    co.name AS country_name,
    cl.name AS climate_name,
    h.name AS hotel_name,
    h.category AS hotel_category,
    tp.duration_weeks,
    tp.price,
    d.type AS discount_type,
    COALESCE(d.percent, 0) AS discount_percent,
    o.departure_date
FROM orders o
JOIN client c ON o.client_id = c.client_id
JOIN travel_package tp ON o.package_id = tp.package_id
JOIN country co ON tp.country_id = co.country_id
JOIN climate cl ON co.climate_id = cl.climate_id
JOIN hotel h ON tp.hotel_id = h.hotel_id
LEFT JOIN discount d ON o.discount_id = d.discount_id
ORDER BY o.order_id;
""")

,order_id,client_full_name,country_name,climate_name,hotel_name,hotel_category,duration_weeks,price,discount_type,discount_percent,departure_date
0,1,Иванова Анна Сергеевна,Турция,Средиземноморский,Sunrise Resort,4,1,65000.0,Постоянный клиент + раннее бронирование,15,2026-07-10
1,2,Иванова Анна Сергеевна,Турция,Средиземноморский,Sunrise Resort,4,1,65000.0,Постоянный клиент + раннее бронирование,15,2026-07-10
2,3,Петров Максим Олегович,Греция,Средиземноморский,Aegean Pearl,5,2,128000.0,Раннее бронирование,10,2026-08-05
3,4,Соколова Елена Андреевна,Таиланд,Тропический,Siam Garden,4,2,165000.0,None,0,2026-09-12
4,5,Кузнецов Игорь Павлович,Финляндия,Умеренный северный,Northern Lights Hotel,3,1,90000.0,Постоянный клиент,5,2026-12-20
5,6,Соколова Елена Андреевна,Греция,Средиземноморский,Aegean Pearl,5,1,78000.0,Семейная покупка,7,2026-06-15


## 8. CRUD-операции через SQLAlchemy ORM

### Create — добавление нового клиента

In [9]:
try:
    session = SessionLocal()
    new_client = Client(
        last_name="Тестова",
        first_name="Мария",
        middle_name="Ивановна",
        address="г. Омск, ул. Учебная, д. 1",
        phone="+7-900-555-66-77",
    )
    session.add(new_client)
    session.commit()
    created_client_id = new_client.client_id
    print(f"Клиент успешно добавлен. ID: {created_client_id}")
except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка при добавлении клиента:")
    print(error)
finally:
    session.close()

Клиент успешно добавлен. ID: 5


### Read — чтение данных о клиентах

In [10]:
try:
    session = SessionLocal()
    clients_list = session.query(Client).order_by(Client.client_id).all()
    for item in clients_list:
        print(item.client_id, item.last_name, item.first_name, item.middle_name, item.phone)
except SQLAlchemyError as error:
    print("Ошибка при чтении данных:")
    print(error)
finally:
    session.close()

1 Иванова Анна Сергеевна +7-900-111-22-33
2 Петров Максим Олегович +7-900-222-33-44
3 Соколова Елена Андреевна +7-900-333-44-55
4 Кузнецов Игорь Павлович +7-900-444-55-66
5 Тестова Мария Ивановна +7-900-555-66-77


### Update — изменение адреса тестового клиента

In [11]:
try:
    session = SessionLocal()
    client = session.query(Client).filter(Client.client_id == created_client_id).first()
    if client:
        client.address = "г. Омск, ул. Обновленная, д. 15"
        session.commit()
        print("Адрес клиента успешно изменен")
    else:
        print("Клиент не найден")
except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка при изменении данных:")
    print(error)
finally:
    session.close()

Адрес клиента успешно изменен


In [12]:
show_query(f"""
SELECT *
FROM client
WHERE client_id = {created_client_id};
""")

,client_id,last_name,first_name,middle_name,address,phone
0,5,Тестова,Мария,Ивановна,"г. Омск, ул. Обновленная, д. 15",+7-900-555-66-77


### Delete — удаление тестового клиента

In [13]:
try:
    session = SessionLocal()
    client = session.query(Client).filter(Client.client_id == created_client_id).first()
    if client:
        session.delete(client)
        session.commit()
        print(f"Клиент с ID {created_client_id} успешно удален")
    else:
        print("Клиент уже удален или не найден")
except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка при удалении клиента:")
    print(error)
finally:
    session.close()

Клиент с ID 5 успешно удален


In [14]:
show_query(f"""
SELECT *
FROM client
WHERE client_id = {created_client_id};
""")

,client_id,last_name,first_name,middle_name,address,phone


## 9. Сложные запросы на выборку данных

### Запрос 1 — доступные варианты путевок

In [15]:
show_query("""
SELECT
    tp.package_id,
    co.name AS country_name,
    cl.name AS climate_name,
    cl.description AS climate_description,
    h.name AS hotel_name,
    h.category AS hotel_category,
    tp.duration_weeks,
    tp.price
FROM travel_package tp
JOIN country co ON tp.country_id = co.country_id
JOIN climate cl ON co.climate_id = cl.climate_id
JOIN hotel h ON tp.hotel_id = h.hotel_id
ORDER BY co.name, h.name, tp.duration_weeks;
""")

,package_id,country_name,climate_name,climate_description,hotel_name,hotel_category,duration_weeks,price
0,3,Греция,Средиземноморский,Мягкая зима и теплое сухое лето,Aegean Pearl,5,1,78000.0
1,4,Греция,Средиземноморский,Мягкая зима и теплое сухое лето,Aegean Pearl,5,2,128000.0
2,5,Таиланд,Тропический,Высокая температура и влажность в течение года,Siam Garden,4,2,165000.0
3,6,Таиланд,Тропический,Высокая температура и влажность в течение года,Siam Garden,4,4,295000.0
4,1,Турция,Средиземноморский,Мягкая зима и теплое сухое лето,Sunrise Resort,4,1,65000.0
5,2,Турция,Средиземноморский,Мягкая зима и теплое сухое лето,Sunrise Resort,4,2,105000.0
6,7,Финляндия,Умеренный северный,Прохладное лето и холодная снежная зима,Northern Lights Hotel,3,1,90000.0


### Запрос 2 — проданные путевки с расчетом итоговой стоимости

In [16]:
show_query("""
SELECT
    o.order_id,
    c.last_name || ' ' || c.first_name || ' ' || COALESCE(c.middle_name, '') AS client_full_name,
    co.name AS country_name,
    h.name AS hotel_name,
    tp.duration_weeks,
    tp.price,
    COALESCE(d.percent, 0) AS discount_percent,
    ROUND(tp.price * (1 - COALESCE(d.percent, 0) / 100.0), 2) AS final_price,
    o.departure_date
FROM orders o
JOIN client c ON o.client_id = c.client_id
JOIN travel_package tp ON o.package_id = tp.package_id
JOIN country co ON tp.country_id = co.country_id
JOIN hotel h ON tp.hotel_id = h.hotel_id
LEFT JOIN discount d ON o.discount_id = d.discount_id
ORDER BY o.order_id;
""")

,order_id,client_full_name,country_name,hotel_name,duration_weeks,price,discount_percent,final_price,departure_date
0,1,Иванова Анна Сергеевна,Турция,Sunrise Resort,1,65000.0,15,55250.0,2026-07-10
1,2,Иванова Анна Сергеевна,Турция,Sunrise Resort,1,65000.0,15,55250.0,2026-07-10
2,3,Петров Максим Олегович,Греция,Aegean Pearl,2,128000.0,10,115200.0,2026-08-05
3,4,Соколова Елена Андреевна,Таиланд,Siam Garden,2,165000.0,0,165000.0,2026-09-12
4,5,Кузнецов Игорь Павлович,Финляндия,Northern Lights Hotel,1,90000.0,5,85500.0,2026-12-20
5,6,Соколова Елена Андреевна,Греция,Aegean Pearl,1,78000.0,7,72540.0,2026-06-15


### Запрос 3 — общая сумма покупок каждого клиента

In [17]:
show_query("""
SELECT
    c.client_id,
    c.last_name || ' ' || c.first_name || ' ' || COALESCE(c.middle_name, '') AS client_full_name,
    COUNT(o.order_id) AS packages_count,
    SUM(ROUND(tp.price * (1 - COALESCE(d.percent, 0) / 100.0), 2)) AS total_payment
FROM client c
JOIN orders o ON c.client_id = o.client_id
JOIN travel_package tp ON o.package_id = tp.package_id
LEFT JOIN discount d ON o.discount_id = d.discount_id
GROUP BY c.client_id, client_full_name
ORDER BY total_payment DESC;
""")

,client_id,client_full_name,packages_count,total_payment
0,3,Соколова Елена Андреевна,2,237540.0
1,2,Петров Максим Олегович,1,115200.0
2,1,Иванова Анна Сергеевна,2,110500.0
3,4,Кузнецов Игорь Павлович,1,85500.0


### Запрос 4 — продажи по странам

In [18]:
show_query("""
SELECT
    co.name AS country_name,
    COUNT(o.order_id) AS sold_packages_count,
    SUM(ROUND(tp.price * (1 - COALESCE(d.percent, 0) / 100.0), 2)) AS total_revenue
FROM orders o
JOIN travel_package tp ON o.package_id = tp.package_id
JOIN country co ON tp.country_id = co.country_id
LEFT JOIN discount d ON o.discount_id = d.discount_id
GROUP BY co.name
ORDER BY total_revenue DESC;
""")

,country_name,sold_packages_count,total_revenue
0,Греция,2,187740.0
1,Таиланд,1,165000.0
2,Турция,2,110500.0
3,Финляндия,1,85500.0


### Запрос 5 — покупки нескольких путевок одним клиентом

In [19]:
show_query("""
SELECT
    c.client_id,
    c.last_name || ' ' || c.first_name || ' ' || COALESCE(c.middle_name, '') AS client_full_name,
    o.departure_date,
    COUNT(o.order_id) AS packages_count
FROM orders o
JOIN client c ON o.client_id = c.client_id
GROUP BY c.client_id, client_full_name, o.departure_date
HAVING COUNT(o.order_id) > 1
ORDER BY packages_count DESC;
""")

,client_id,client_full_name,departure_date,packages_count
0,1,Иванова Анна Сергеевна,2026-07-10,2


### Запрос 6 — статистика продаж по отелям

In [20]:
show_query("""
SELECT
    h.hotel_id,
    h.name AS hotel_name,
    h.category,
    COUNT(o.order_id) AS sold_packages_count,
    SUM(ROUND(tp.price * (1 - COALESCE(d.percent, 0) / 100.0), 2)) AS total_revenue
FROM hotel h
JOIN travel_package tp ON h.hotel_id = tp.hotel_id
JOIN orders o ON tp.package_id = o.package_id
LEFT JOIN discount d ON o.discount_id = d.discount_id
GROUP BY h.hotel_id, h.name, h.category
ORDER BY total_revenue DESC;
""")

,hotel_id,hotel_name,category,sold_packages_count,total_revenue
0,2,Aegean Pearl,5,2,187740.0
1,3,Siam Garden,4,1,165000.0
2,1,Sunrise Resort,4,2,110500.0
3,4,Northern Lights Hotel,3,1,85500.0


## 10. Создание представления

Представление `order_payment_view` объединяет сведения о заказах и рассчитывает итоговую стоимость каждой проданной путевки.

In [21]:
try:
    with engine.begin() as connection:
        connection.execute(text("DROP VIEW IF EXISTS order_payment_view;"))
        connection.execute(text("""
            CREATE VIEW order_payment_view AS
            SELECT
                o.order_id,
                o.client_id,
                c.last_name || ' ' || c.first_name || ' ' || COALESCE(c.middle_name, '') AS client_full_name,
                o.package_id,
                co.name AS country_name,
                cl.name AS climate_name,
                h.name AS hotel_name,
                h.category AS hotel_category,
                tp.duration_weeks,
                tp.price,
                o.discount_id,
                d.type AS discount_type,
                COALESCE(d.percent, 0) AS discount_percent,
                ROUND(tp.price * (1 - COALESCE(d.percent, 0) / 100.0), 2) AS final_price,
                o.departure_date
            FROM orders o
            JOIN client c ON o.client_id = c.client_id
            JOIN travel_package tp ON o.package_id = tp.package_id
            JOIN country co ON tp.country_id = co.country_id
            JOIN climate cl ON co.climate_id = cl.climate_id
            JOIN hotel h ON tp.hotel_id = h.hotel_id
            LEFT JOIN discount d ON o.discount_id = d.discount_id;
        """))
    print("Представление order_payment_view успешно создано")
except SQLAlchemyError as error:
    print("Ошибка при создании представления:")
    print(error)

Представление order_payment_view успешно создано


In [22]:
show_query("""
SELECT *
FROM order_payment_view
ORDER BY order_id;
""")

,order_id,client_id,client_full_name,package_id,country_name,climate_name,hotel_name,hotel_category,duration_weeks,price,discount_id,discount_type,discount_percent,final_price,departure_date
0,1,1,Иванова Анна Сергеевна,1,Турция,Средиземноморский,Sunrise Resort,4,1,65000.0,3.0,Постоянный клиент + раннее бронирование,15,55250.0,2026-07-10
1,2,1,Иванова Анна Сергеевна,1,Турция,Средиземноморский,Sunrise Resort,4,1,65000.0,3.0,Постоянный клиент + раннее бронирование,15,55250.0,2026-07-10
2,3,2,Петров Максим Олегович,4,Греция,Средиземноморский,Aegean Pearl,5,2,128000.0,2.0,Раннее бронирование,10,115200.0,2026-08-05
3,4,3,Соколова Елена Андреевна,5,Таиланд,Тропический,Siam Garden,4,2,165000.0,NaN,None,0,165000.0,2026-09-12
4,5,4,Кузнецов Игорь Павлович,7,Финляндия,Умеренный северный,Northern Lights Hotel,3,1,90000.0,1.0,Постоянный клиент,5,85500.0,2026-12-20
5,6,3,Соколова Елена Андреевна,3,Греция,Средиземноморский,Aegean Pearl,5,1,78000.0,4.0,Семейная покупка,7,72540.0,2026-06-15


### Общая сумма покупок каждого клиента через представление

In [23]:
show_query("""
SELECT
    client_id,
    client_full_name,
    COUNT(order_id) AS packages_count,
    SUM(final_price) AS total_payment
FROM order_payment_view
GROUP BY client_id, client_full_name
ORDER BY total_payment DESC;
""")

,client_id,client_full_name,packages_count,total_payment
0,3,Соколова Елена Андреевна,2,237540.0
1,2,Петров Максим Олегович,1,115200.0
2,1,Иванова Анна Сергеевна,2,110500.0
3,4,Кузнецов Игорь Павлович,1,85500.0


## 11. Хранимая функция расчета итоговой стоимости

In [24]:
try:
    with engine.begin() as connection:
        connection.execute(text("DROP FUNCTION IF EXISTS calculate_order_total(NUMERIC, INTEGER);"))
        connection.execute(text("""
            CREATE OR REPLACE FUNCTION calculate_order_total(
                p_price NUMERIC,
                p_discount_percent INTEGER
            )
            RETURNS NUMERIC
            LANGUAGE plpgsql
            AS $$
            BEGIN
                IF p_price <= 0 THEN
                    RAISE EXCEPTION 'Стоимость путевки должна быть положительной';
                END IF;

                IF COALESCE(p_discount_percent, 0) NOT BETWEEN 0 AND 100 THEN
                    RAISE EXCEPTION 'Размер скидки должен находиться в диапазоне от 0 до 100';
                END IF;

                RETURN ROUND(p_price * (1 - COALESCE(p_discount_percent, 0) / 100.0), 2);
            END;
            $$;
        """))
    print("Функция calculate_order_total успешно создана")
except SQLAlchemyError as error:
    print("Ошибка при создании функции:")
    print(error)

Функция calculate_order_total успешно создана


In [25]:
show_query("""
SELECT calculate_order_total(100000, 15) AS payment_result;
""")

,payment_result
0,85000.0


In [26]:
show_query("""
SELECT
    order_id,
    client_full_name,
    price,
    discount_percent,
    calculate_order_total(price, discount_percent) AS final_price
FROM order_payment_view
ORDER BY order_id;
""")

,order_id,client_full_name,price,discount_percent,final_price
0,1,Иванова Анна Сергеевна,65000.0,15,55250.0
1,2,Иванова Анна Сергеевна,65000.0,15,55250.0
2,3,Петров Максим Олегович,128000.0,10,115200.0
3,4,Соколова Елена Андреевна,165000.0,0,165000.0
4,5,Кузнецов Игорь Павлович,90000.0,5,85500.0
5,6,Соколова Елена Андреевна,78000.0,7,72540.0


## 12. Хранимая процедура назначения скидки заказу

In [27]:
try:
    with engine.begin() as connection:
        connection.execute(text("DROP PROCEDURE IF EXISTS assign_discount_to_order(BIGINT, BIGINT);"))
        connection.execute(text("""
            CREATE OR REPLACE PROCEDURE assign_discount_to_order(
                p_order_id BIGINT,
                p_discount_id BIGINT
            )
            LANGUAGE plpgsql
            AS $$
            BEGIN
                IF NOT EXISTS (
                    SELECT 1 FROM discount WHERE discount_id = p_discount_id
                ) THEN
                    RAISE EXCEPTION 'Скидка с ID % не найдена', p_discount_id;
                END IF;

                UPDATE orders
                SET discount_id = p_discount_id
                WHERE order_id = p_order_id;

                IF NOT FOUND THEN
                    RAISE EXCEPTION 'Заказ с ID % не найден', p_order_id;
                END IF;
            END;
            $$;
        """))
    print("Процедура assign_discount_to_order успешно создана")
except SQLAlchemyError as error:
    print("Ошибка при создании процедуры:")
    print(error)

Процедура assign_discount_to_order успешно создана


In [28]:
print("Заказ до изменения скидки")
display(show_query("""
SELECT order_id, client_full_name, discount_type, discount_percent, final_price
FROM order_payment_view
WHERE order_id = 4;
"""))

try:
    with engine.begin() as connection:
        connection.execute(text("CALL assign_discount_to_order(4, 2);"))
    print("Скидка заказу успешно назначена")
except SQLAlchemyError as error:
    print("Ошибка при вызове процедуры:")
    print(error)

print("Заказ после изменения скидки")
display(show_query("""
SELECT order_id, client_full_name, discount_type, discount_percent, final_price
FROM order_payment_view
WHERE order_id = 4;
"""))

Заказ до изменения скидки


,order_id,client_full_name,discount_type,discount_percent,final_price
0,4,Соколова Елена Андреевна,None,0,165000.0


Скидка заказу успешно назначена
Заказ после изменения скидки


,order_id,client_full_name,discount_type,discount_percent,final_price
0,4,Соколова Елена Андреевна,Раннее бронирование,10,148500.0


## 13. Триггер контроля соответствия отеля стране

В предоставленной физической модели таблица `hotel` не содержит внешний ключ на страну. Триггер запрещает использовать один и тот же отель в путевках для разных стран.

In [29]:
try:
    with engine.begin() as connection:
        connection.execute(text("DROP TRIGGER IF EXISTS check_hotel_country_trigger ON travel_package;"))
        connection.execute(text("DROP FUNCTION IF EXISTS check_hotel_country_consistency();"))
        connection.execute(text("""
            CREATE OR REPLACE FUNCTION check_hotel_country_consistency()
            RETURNS TRIGGER
            LANGUAGE plpgsql
            AS $$
            BEGIN
                IF EXISTS (
                    SELECT 1
                    FROM travel_package
                    WHERE hotel_id = NEW.hotel_id
                      AND country_id <> NEW.country_id
                      AND package_id <> COALESCE(NEW.package_id, 0)
                ) THEN
                    RAISE EXCEPTION
                        'Один отель не может относиться к разным странам. ID отеля: %',
                        NEW.hotel_id;
                END IF;

                RETURN NEW;
            END;
            $$;
        """))
        connection.execute(text("""
            CREATE TRIGGER check_hotel_country_trigger
            BEFORE INSERT OR UPDATE ON travel_package
            FOR EACH ROW
            EXECUTE FUNCTION check_hotel_country_consistency();
        """))
    print("Триггер check_hotel_country_trigger успешно создан")
except SQLAlchemyError as error:
    print("Ошибка при создании триггера:")
    print(error)

Триггер check_hotel_country_trigger успешно создан


### Проверка работы триггера

Пытаемся добавить путевку в Грецию с отелем, который уже используется для Турции. Ошибка является ожидаемым результатом.

In [30]:
try:
    with engine.begin() as connection:
        connection.execute(text("""
            INSERT INTO travel_package (country_id, hotel_id, price, duration_weeks)
            VALUES (2, 1, 99000, 4);
        """))
    print("Тестовая путевка добавлена")
except SQLAlchemyError as error:
    print("Триггер сработал корректно:")
    print(error)

Триггер сработал корректно:
(psycopg2.errors.RaiseException) ОШИБКА:  Один отель не может относиться к разным странам. ID отеля: 1
CONTEXT:  функция PL/pgSQL check_hotel_country_consistency(), строка 10, оператор RAISE

[SQL: 
            INSERT INTO travel_package (country_id, hotel_id, price, duration_weeks)
            VALUES (2, 1, 99000, 4);
        ]
(Background on this error at: https://sqlalche.me/e/20/2j85)


## 14. Обработка исключений

### Ошибка: недопустимая длительность тура

In [31]:
try:
    session = SessionLocal()
    wrong_package = TravelPackage(
        country_id=1,
        hotel_id=1,
        price=Decimal("50000.00"),
        duration_weeks=3,
    )
    session.add(wrong_package)
    session.commit()
    print("Путевка добавлена")
except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка перехвачена корректно:")
    print(error)
finally:
    session.close()

Ошибка перехвачена корректно:
(psycopg2.errors.CheckViolation) ОШИБКА:  новая строка в отношении "travel_package" нарушает ограничение-проверку "check_package_duration"
DETAIL:  Ошибочная строка содержит (9, 1, 1, 50000.00, 3).

[SQL: INSERT INTO travel_package (country_id, hotel_id, price, duration_weeks) VALUES (%(country_id)s, %(hotel_id)s, %(price)s, %(duration_weeks)s) RETURNING travel_package.package_id]
[parameters: {'country_id': 1, 'hotel_id': 1, 'price': Decimal('50000.00'), 'duration_weeks': 3}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


### Ошибка: некорректная категория отеля

In [32]:
try:
    session = SessionLocal()
    wrong_hotel = Hotel(name="Ошибочный отель", category=8, address="Тестовый адрес")
    session.add(wrong_hotel)
    session.commit()
    print("Отель добавлен")
except SQLAlchemyError as error:
    session.rollback()
    print("Ошибка перехвачена корректно:")
    print(error)
finally:
    session.close()

Ошибка перехвачена корректно:
(psycopg2.errors.CheckViolation) ОШИБКА:  новая строка в отношении "hotel" нарушает ограничение-проверку "check_hotel_category"
DETAIL:  Ошибочная строка содержит (5, Ошибочный отель, 8, Тестовый адрес).

[SQL: INSERT INTO hotel (name, category, address) VALUES (%(name)s, %(category)s, %(address)s) RETURNING hotel.hotel_id]
[parameters: {'name': 'Ошибочный отель', 'category': 8, 'address': 'Тестовый адрес'}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


### Ошибка: назначение несуществующей скидки

In [33]:
try:
    with engine.begin() as connection:
        connection.execute(text("CALL assign_discount_to_order(1, 999);"))
    print("Скидка назначена")
except SQLAlchemyError as error:
    print("Ошибка перехвачена корректно:")
    print(error)

Ошибка перехвачена корректно:
(psycopg2.errors.RaiseException) ОШИБКА:  Скидка с ID 999 не найдена
CONTEXT:  функция PL/pgSQL assign_discount_to_order(bigint,bigint), строка 6, оператор RAISE

[SQL: CALL assign_discount_to_order(1, 999);]
(Background on this error at: https://sqlalche.me/e/20/2j85)


## 15. Итоговая проверка данных

In [34]:
for table_name, order_column in tables.items():
    print(f"\nТаблица: {table_name}")
    display(show_query(f"SELECT * FROM {table_name} ORDER BY {order_column};"))


Таблица: client


,client_id,last_name,first_name,middle_name,address,phone
0,1,Иванова,Анна,Сергеевна,"г. Омск, ул. Ленина, д. 10",+7-900-111-22-33
1,2,Петров,Максим,Олегович,"г. Омск, ул. Мира, д. 25",+7-900-222-33-44
2,3,Соколова,Елена,Андреевна,"г. Омск, пр. Карла Маркса, д. 8",+7-900-333-44-55
3,4,Кузнецов,Игорь,Павлович,"г. Омск, ул. Герцена, д. 17",+7-900-444-55-66



Таблица: climate


,climate_id,name,description
0,1,Средиземноморский,Мягкая зима и теплое сухое лето
1,2,Тропический,Высокая температура и влажность в течение года
2,3,Умеренный северный,Прохладное лето и холодная снежная зима



Таблица: country


...

,package_id,country_id,hotel_id,price,duration_weeks
0,1,1,1,65000.0,1
1,2,1,1,105000.0,2
2,3,2,2,78000.0,1
3,4,2,2,128000.0,2
4,5,3,3,165000.0,2
5,6,3,3,295000.0,4
6,7,4,4,90000.0,1



Таблица: discount


,discount_id,type,percent
0,1,Постоянный клиент,5
1,2,Раннее бронирование,10
2,3,Постоянный клиент + раннее бронирование,15
3,4,Семейная покупка,7
4,5,Семейная покупка + раннее бронирование,17



Таблица: orders


,order_id,client_id,package_id,discount_id,departure_date
0,1,1,1,3,2026-07-10
1,2,1,1,3,2026-07-10
2,3,2,4,2,2026-08-05
3,4,3,5,2,2026-09-12
4,5,4,7,1,2026-12-20
5,6,3,3,4,2026-06-15


### Итоговый расчет стоимости проданных путевок

In [35]:
show_query("""
SELECT
    order_id,
    client_full_name,
    country_name,
    hotel_name,
    duration_weeks,
    price,
    discount_type,
    discount_percent,
    final_price,
    departure_date
FROM order_payment_view
ORDER BY order_id;
""")

,order_id,client_full_name,country_name,hotel_name,duration_weeks,price,discount_type,discount_percent,final_price,departure_date
0,1,Иванова Анна Сергеевна,Турция,Sunrise Resort,1,65000.0,Постоянный клиент + раннее бронирование,15,55250.0,2026-07-10
1,2,Иванова Анна Сергеевна,Турция,Sunrise Resort,1,65000.0,Постоянный клиент + раннее бронирование,15,55250.0,2026-07-10
2,3,Петров Максим Олегович,Греция,Aegean Pearl,2,128000.0,Раннее бронирование,10,115200.0,2026-08-05
3,4,Соколова Елена Андреевна,Таиланд,Siam Garden,2,165000.0,Раннее бронирование,10,148500.0,2026-09-12
4,5,Кузнецов Игорь Павлович,Финляндия,Northern Lights Hotel,1,90000.0,Постоянный клиент,5,85500.0,2026-12-20
5,6,Соколова Елена Андреевна,Греция,Aegean Pearl,1,78000.0,Семейная покупка,7,72540.0,2026-06-15


### Общая выручка туристической фирмы

In [36]:
show_query("""
SELECT SUM(final_price) AS total_company_revenue
FROM order_payment_view;
""")

,total_company_revenue
0,532240.0


## Вывод

В ходе выполнения работы была реализована реляционная база данных туристической фирмы. Созданы ORM-модели, выполнены CRUD-операции, сложные запросы, представление, хранимая функция, процедура и триггер. Также продемонстрирована обработка исключений при нарушении ограничений базы данных.